<a href="https://colab.research.google.com/github/cbonnin88/Python-For-Product/blob/main/PM_Feedback_Radar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 15.9 MB/s eta 0:00:00


In [5]:
import polars as pl
import streamlit as st

In [9]:
# 1. Page Configuration
st.set_page_config(page_title='PM App Review Analyze', layout='wide')
st.title('📱 Product Manager: App Review Analyzer')
st.write('Analyze user feedback, identify bug reports, and discover feature requests')

# 2. Load Data with Polars
@st.cache_data
def load_data(filepath: str)-> pl.DataFrame:
  try:
    df_product = pl.read_csv(filepath)
    return df_product
  except Exception as e:
    st.error(f'Error loading file: {e}. Please make sure health_fitness_app_reviews.csv is uploaded to your Colab workspace!')
    return pl.DataFrame()

df_product = load_data('health_fitness_app_reviews.csv')

# Stop executing if the dataframe is empty (file not found)
if df_product.is_empty():
  st.stop()

# 3. Sidebar Filters
st.sidebar.header('🔍 Filter Reviews')

# Get unique app names
all_apps = df_product['appName'].unique().sort().to_list()
selected_apps_filter = st.sidebar.multiselect('Select App(s):',options=all_apps,default=all_apps)

# Get unique category apps
all_tags = df_product['tag'].unique().sort().to_list()
selected_tags_filter = st.sidebar.multiselect('Select Category Tag(s):',options=all_tags,default=all_tags)

# Star rating filter
selected_scores = st.sidebar.multiselect('Select Star Ratings:',options=[1, 2, 3, 4, 5],default=[1, 2, 3, 4, 5])

# Keyword search for finding pain points
search_keyword = st.sidebar.text_input('Search keyword in reviews (e.g. crash, price, bug):','')

# 4. Filter the DataFrame
filtered_product_df = df_product.filter(
    (pl.col('appName').is_in(selected_apps_filter)) &
    (pl.col('tag').is_in(selected_tags_filter)) &
    (pl.col('score').is_in(selected_scores))
)

# Apply keyword filter if you typed somethihng in the search box
if search_keyword:
  filtered_product_df = filtered_product_df.filter(
      pl.col('content').str.to_lowercase().str.contains(search_keyword.lower())
  )

# 5. KPI Metrics Section
st.subheader('📊 Key Performance Indicators')
col1,col2,col3,col4 = st.columns(4)

total_reviews = filtered_product_df.height
avg_score = filtered_product_df['score'].mean() if total_reviews > 0 else 0.0
negative_reviews = filtered_product_df.filter(pl.col('score')<=2).height
neg_percentage = (negative_reviews / total_reviews * 100) if total_reviews > 0 else 0.0

with col1:
  st.metric(label='Total Reviews',value=f'{total_reviews:,}')
with col2:
  st.metric(label='Average Star Rating',value=f'{avg_score:.2f} ⭐')
with col3:
  st.metric(label='Critical Reviews (1-2 ⭐)',value=f'{negative_reviews:,}')
with col4:
  st.metric(label='Negative Feedback %',value=f'{neg_percentage:.1f}%')

st.markdown('---')

# 6. Deep Dive into User Voice (Sorted by Thumbs Up Count)
st.subheader('🗣️ Top User Feedback (Sorted by Impact)')
st.write('Sorting by `thumbsUpCount` shows which reviews resonate most with the broader user community—crucial for prioritizing bug fixes!')

# Sort
sorted_df = filtered_product_df.sort(by='thumbsUpCount',descending=True)

# Displaying tabular data
st.dataframe(
    sorted_df.select(["appName", "score", "thumbsUpCount", "content", "at"]),
    width='stretch',
    height=400
)

# 7. Quick PM Insights: Score Distribution
st.subheader('📈 Rating Distribution')
if total_reviews > 0:
  score_dist = filtered_product_df.group_by('score').len().sort('score')
  st.bar_chart(score_dist.to_pandas().set_index('score'))
else:
  st.info('No Data available for the selected filters.')

2026-07-24 08:46:49.445 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-24 08:46:49.447 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-24 08:46:49.448 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-24 08:46:49.449 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-24 08:46:49.450 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-24 08:46:49.451 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-24 08:46:49.453 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-24 08:46:49.454 No runtime found, using MemoryCacheStorageManager
2026-07-24 08:46:49.704 Thread 'MainThread':